# Dataset B – Wine Quality


In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import StratifiedKFold, learning_curve, cross_validate
from sklearn.metrics import confusion_matrix

SEED = 49
np.random.seed(SEED)
random.seed(SEED)

### Data Loading

We load the Wine Quality dataset and inspect its basic structure.

In [3]:
DATA_PATH = "../wine.csv"
df = pd.read_csv(DATA_PATH)

df.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,class,type,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0,5


The variables quality, class, and type are categorical in nature, despite being encoded as integers. Specifically, quality is an ordinal multi-class target variable, type is a binary categorical feature, and class represents a discrete categorical grouping. All remaining features are continuous numerical physicochemical measurements.

## Experiments

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

X = df.drop(columns=["quality", "class"])
y = df["quality"]

num_cols = ["fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar", 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'pH', 'density', 'sulphates', 'alcohol']
cat_cols = ["type"]

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)


In [5]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20
VAL_SIZE  = 0.10

y_arr = pd.Series(y).values

X_train_t, X_test, y_train_t, y_test = train_test_split(
    X, y_arr, test_size=TEST_SIZE, stratify=y_arr, random_state=SEED
)

val_size_within_train = VAL_SIZE / (1.0 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_t, y_train_t, test_size=val_size_within_train, stratify=y_train_t, random_state=SEED
)

### Neural Network

#### PyTorch + SVD

In [6]:
import torch

X_train_sp = preprocessor.fit_transform(X_train)
X_val_sp   = preprocessor.transform(X_val)
X_test_sp  = preprocessor.transform(X_test)

X_train_nn = X_train_sp.toarray() if hasattr(X_train_sp, "toarray") else np.asarray(X_train_sp)
X_val_nn   = X_val_sp.toarray()   if hasattr(X_val_sp, "toarray")   else np.asarray(X_val_sp)
X_test_nn  = X_test_sp.toarray()  if hasattr(X_test_sp, "toarray")  else np.asarray(X_test_sp)

X_train_nn = X_train_nn.astype(np.float32, copy=False)
X_val_nn   = X_val_nn.astype(np.float32, copy=False)
X_test_nn  = X_test_nn.astype(np.float32, copy=False)

y_train_s = np.asarray(y_train)
y_val_s   = np.asarray(y_val)
y_test_s  = np.asarray(y_test)

classes = np.unique(np.concatenate([y_train_s, y_val_s, y_test_s]))
class_to_idx = {c: i for i, c in enumerate(classes)}

y_train_idx = np.vectorize(class_to_idx.get)(y_train_s).astype(np.int64)
y_val_idx   = np.vectorize(class_to_idx.get)(y_val_s).astype(np.int64)
y_test_idx  = np.vectorize(class_to_idx.get)(y_test_s).astype(np.int64)

X_train_nn_t = torch.from_numpy(X_train_nn)
X_val_nn_t   = torch.from_numpy(X_val_nn)
X_test_nn_t  = torch.from_numpy(X_test_nn)

y_train_nn = torch.tensor(y_train_idx, dtype=torch.long)
y_val_nn   = torch.tensor(y_val_idx,   dtype=torch.long)
y_test_nn  = torch.tensor(y_test_idx,  dtype=torch.long)

input_dim = X_train_nn_t.shape[1]
num_classes = len(classes)

In [7]:
import torch
import torch.nn as nn

class MLP_tmp(nn.Module):
    def __init__(self, d, h, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(d, h),
            nn.ReLU()
        )
        self.classifier = nn.Linear(h, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x))

    def freeze_all(self):
        for p in self.parameters():
            p.requires_grad = False

    def unfreeze_last_k(self, k=1):
        self.freeze_all()
        for p in self.classifier.parameters():
            p.requires_grad = True
        if k >= 2:
            for m in self.features:
                if isinstance(m, nn.Linear):
                    for p in m.parameters():
                        p.requires_grad = True
    def count_trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [9]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score
import gc
import time


gc.collect()
torch.manual_seed(SEED)
np.random.seed(SEED)


BATCH_SIZE = 64

train_ds = TensorDataset(X_train_nn_t, y_train_nn)
val_ds   = TensorDataset(X_val_nn_t,   y_val_nn)
test_ds  = TensorDataset(X_test_nn_t,  y_test_nn)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MLP_tmp(d=input_dim, h=32, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def eval_f1(model, loader):
    """
    Compute Macro-F1 score for a PyTorch model

    Input:
        model (torch.nn.Module): trained neural network model
        loader (DataLoader): DataLoader providing evaluation batches
    Return:float: Macro-F1 score computed on the provided dataset
    """
    preds, ys = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(torch.argmax(model(xb), dim=1).cpu().numpy())
            ys.append(yb.cpu().numpy())

    return f1_score(np.concatenate(ys), np.concatenate(preds), average="macro")


def eval_loss(model, loader):
    total, n = 0.0, 0
    model.eval()
    
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = criterion(model(xb), yb)
            bs = yb.size(0)
            total += loss.item() * bs
            n += bs
    return total / n


grad_evals = 0
t0 = time.time()

for epoch in range(10):  
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        grad_evals += 1
    vloss = eval_loss(model, val_loader)
    vf1   = eval_f1(model, val_loader)
    print(f"epoch {epoch:02d} | val_loss={vloss:.4f} | val_f1={vf1:.4f} | grad_evals={grad_evals} | time={time.time()-t0:.1f}s")


epoch 00 | val_loss=1.5660 | val_f1=0.2216 | grad_evals=72 | time=0.2s
epoch 01 | val_loss=1.2895 | val_f1=0.2575 | grad_evals=144 | time=0.4s
epoch 02 | val_loss=1.2001 | val_f1=0.2653 | grad_evals=216 | time=0.6s
epoch 03 | val_loss=1.1518 | val_f1=0.2794 | grad_evals=288 | time=0.7s
epoch 04 | val_loss=1.1240 | val_f1=0.2708 | grad_evals=360 | time=0.9s
epoch 05 | val_loss=1.1028 | val_f1=0.2973 | grad_evals=432 | time=1.0s
epoch 06 | val_loss=1.0876 | val_f1=0.3116 | grad_evals=504 | time=1.2s
epoch 07 | val_loss=1.0773 | val_f1=0.3256 | grad_evals=576 | time=1.4s
epoch 08 | val_loss=1.0663 | val_f1=0.3368 | grad_evals=648 | time=1.6s
epoch 09 | val_loss=1.0608 | val_f1=0.3275 | grad_evals=720 | time=1.7s


In [10]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

@torch.no_grad()
def evaluate_multiclass(model, loader, device="cpu", average="macro"):
    model.eval()
    preds, ys = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        preds.append(torch.argmax(logits, dim=1).cpu().numpy())
        ys.append(yb.cpu().numpy())
    y_pred = np.concatenate(preds)
    y_true = np.concatenate(ys)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average=average),
        "cm": confusion_matrix(y_true, y_pred),
        "y_pred": y_pred
    }

In [11]:
res = evaluate_multiclass(model, test_loader, device=device, average="macro")
print("Accuracy:", res["accuracy"])
print("Macro-F1:", res["f1"])
print("Confusion matrix:\n", res["cm"])

Accuracy: 0.5392307692307692
Macro-F1: 0.3022376892258609
Confusion matrix:
 [[  0   0   2   2   0   0   0   0]
 [  0   0  20  13   0   0   0   0]
 [  0   0 167 123   3   0   0   0]
 [  0   0 105 305  37   3   0   0]
 [  0   0  17 120 131  44   0   0]
 [  0   0   0  24  44  92   3   0]
 [  0   0   0   1   3  31   6   0]
 [  0   0   0   0   0   3   1   0]]
